# Treinamento temporal dos modelos

- Treino final: 2014–2019 (7.723.448 casos)
- Validação, Optuna e early stopping: 2020 (1.331.664 casos)
- Teste lacrado: 2021 (não é carregado neste notebook)

A MLP ajusta encoder e medianas somente no treino. XGBoost roda na GPU (`device="cuda"`) e LightGBM em CPU; ambos preservam `NaN` nativamente. O fit final usa todo o treino; o Optuna afina numa amostra estratificada.

In [5]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('../..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from dengue_pipeline.paths import TRAIN_YEARS, VALIDATION_YEARS, TEST_YEARS


N_TRIALS            = 200                  # trials de Optuna (XGBoost/LightGBM)
TUNING_SAMPLE_SIZE  = 400_000             # amostra estratificada para o tuning
MAX_EPOCHS          = 150                 # teto de épocas da MLP (early stopping corta antes)

MLP_HIDDEN          = "1024,512,256,128"  # larguras das camadas ocultas (vírgula separa)
MLP_LEARNING_RATE   = 1e-4
MLP_BATCH_SIZE      = 16_384
MLP_DROPOUT         = 0.2
MLP_PATIENCE        = 10

DEVICE              = "cuda"              # MLP e XGBoost: "cuda" ou "cpu"
LGBM_DEVICE         = "cpu"               # LightGBM: "cpu" ou "gpu" (GPU nem sempre é mais rápido)

print("Split temporal:", TRAIN_YEARS, "| val", VALIDATION_YEARS, "| teste", TEST_YEARS)

Split temporal: (2017, 2018, 2019) | val (2020,) | teste (2021,)


In [6]:
# Monta o comando com as variáveis do cell de config e dispara o treino.
cmd = (
    "../../scripts/train_models.py "
    f"--n-trials {N_TRIALS} --tuning-sample-size {TUNING_SAMPLE_SIZE} --max-epochs {MAX_EPOCHS} "
    f"--mlp-hidden {MLP_HIDDEN} --mlp-learning-rate {MLP_LEARNING_RATE} "
    f"--mlp-batch-size {MLP_BATCH_SIZE} --mlp-dropout {MLP_DROPOUT} --mlp-patience {MLP_PATIENCE} "
    f"--device {DEVICE} --lgbm-device {LGBM_DEVICE}"
)
print("Executando:", cmd)
get_ipython().run_line_magic("run", cmd)

Executando: ../../scripts/train_models.py --n-trials 200 --tuning-sample-size 400000 --max-epochs 150 --mlp-hidden 1024,512,256,128 --mlp-learning-rate 0.0001 --mlp-batch-size 16384 --mlp-dropout 0.2 --mlp-patience 10 --device cuda --lgbm-device cpu
Loading training years: (2017, 2018, 2019)
Loading validation years: (2020,)
Época 001 | treino: 0.40675 | val_loss: 0.43314 | val_PR-AUC: 0.85902 | lr: 1.00e-04
Época 002 | treino: 0.37526 | val_loss: 0.42607 | val_PR-AUC: 0.86506 | lr: 1.00e-04
Época 003 | treino: 0.36828 | val_loss: 0.42244 | val_PR-AUC: 0.86767 | lr: 1.00e-04
Época 004 | treino: 0.36405 | val_loss: 0.42120 | val_PR-AUC: 0.86847 | lr: 1.00e-04
Época 005 | treino: 0.36099 | val_loss: 0.42061 | val_PR-AUC: 0.86893 | lr: 1.00e-04
Época 006 | treino: 0.35854 | val_loss: 0.41911 | val_PR-AUC: 0.87032 | lr: 1.00e-04
Época 007 | treino: 0.35642 | val_loss: 0.42000 | val_PR-AUC: 0.86913 | lr: 1.00e-04
Época 008 | treino: 0.35464 | val_loss: 0.41904 | val_PR-AUC: 0.87048 | lr: 1.

Best trial: 177. Best value: 0.875569: 100%|██████████| 200/200 [21:12<00:00,  6.36s/it]


Melhor average_precision: 0.8756
Melhores parâmetros: {'n_estimators': 400, 'learning_rate': 0.052109957341725584, 'max_depth': 6, 'subsample': 0.7334762192492745, 'colsample_bytree': 0.9140040061131207, 'reg_alpha': 7.2661463868911e-08, 'reg_lambda': 3.6671056931099913e-06, 'min_child_weight': 7}
Optuna usando amostra estratificada de 400,000 registros.
Optuna usando amostra estratificada de 400,000 registros.


Best trial: 89. Best value: 0.875448: 100%|██████████| 200/200 [37:24<00:00, 11.22s/it]


Melhor average_precision: 0.8754
Melhores parâmetros: {'n_estimators': 500, 'learning_rate': 0.033380131349783446, 'max_depth': 7, 'subsample': 0.9203620394872019, 'colsample_bytree': 0.8673256690631755, 'reg_alpha': 3.294598980241675e-07, 'reg_lambda': 2.2361812927766835e-05, 'num_leaves': 102, 'min_child_samples': 90}
Model manifest written to C:\Users\angej\Documents\trabalho_foda\IML-Dengue\artifacts\models\model_manifest.json
